In [11]:
# ###########################################################################
# Author: Edward E. Daisey
# Course: Modeling & Simulation of Complex Systems
# Title: Kuramoto Model
# Date: 31st of March 2026
# ###########################################################################


################################### Overview ##################################
# Description:
#   This script simulates the Kuramoto model from equation (3.1)
#
#       thetaDot_i = omega_i + (K / N) * sum_j sin(theta_j - theta_i)
#
#   by using the order-parameter form from equation (3.3)
#
#       thetaDot_i = omega_i + K * r * sin(psi - theta_i).
#
#   To avoid computing psi explicitly, the code uses the exact identity
#
#       r * sin(psi - theta_i) = S * cos(theta_i) - C * sin(theta_i),
#
#   where
#
#       C = (1 / N) * sum_j cos(theta_j),
#       S = (1 / N) * sum_j sin(theta_j).
#
#   This keeps the algorithm O(N) per time step, as required.
# ###########################################################################


################################### Constants #################################
pi = 3.1415926535897932384626433832795
twoPi = 2.0 * pi
halfPi = 0.5 * pi

oscillatorCount = 10000 # > 1024

timeStep = 0.1
timeStart = 0.0
timeEnd = 20.0
stepCount = int((timeEnd - timeStart) / timeStep)

couplingStart = 0.0
couplingEnd = 10.0
couplingStep = 0.1

frequencyMin = 0.5 * pi
frequencyMax = 3.0 * pi

randomSeed = 10000

dataFilename = "kuramotoData.csv"
figureFilename = "kuramotoFigure.svg"
# ###########################################################################


############################### Function 1 ####################################
# Name:
#   SetSeed
#
# Purpose:
#   Initialize the Mersenne Twister MT19937 pseudo-random number generator.
#
# Input:
#   seedValue : Integer seed.
#
# Output:
#   Updates the global MT state.
mtState = [0] * 624
mtIndex = 624

def SetSeed(seedValue):
    global mtState, mtIndex

    mtState[0] = int(seedValue) & 0xffffffff

    for index in range(1, 624):
        previousValue = mtState[index - 1]
        mtState[index] = (
            1812433253 * (previousValue ^ (previousValue >> 30)) + index
        ) & 0xffffffff

    mtIndex = 624
# ###########################################################################


############################### Function 2 ####################################
# Name:
#   Twist
#
# Purpose:
#   Refresh the MT19937 state array.
#
# Input:
#   None.
#
# Output:
#   Updates the global MT state.
def Twist():
    global mtState, mtIndex

    for index in range(624):
        upperPart = mtState[index] & 0x80000000
        lowerPart = mtState[(index + 1) % 624] & 0x7fffffff
        combinedValue = upperPart + lowerPart

        twistedValue = mtState[(index + 397) % 624] ^ (combinedValue >> 1)

        if combinedValue % 2 != 0:
            twistedValue ^= 0x9908b0df

        mtState[index] = twistedValue & 0xffffffff

    mtIndex = 0
# ###########################################################################


############################### Function 3 ####################################
# Name:
#   ExtractUInt32
#
# Purpose:
#   Extract one 32-bit unsigned integer from MT19937.
#
# Input:
#   None.
#
# Output:
#   A 32-bit unsigned integer.
def ExtractUInt32():
    global mtState, mtIndex

    if mtIndex >= 624:
        Twist()

    value = mtState[mtIndex]
    mtIndex += 1

    value ^= (value >> 11)
    value ^= (value << 7) & 0x9d2c5680
    value ^= (value << 15) & 0xefc60000
    value ^= (value >> 18)

    return value & 0xffffffff
# ###########################################################################


############################### Function 4 ####################################
# Name:
#   RandomUnit
#
# Purpose:
#   Return a pseudo-random number in [0, 1) using MT19937.
#
# Input:
#   None.
#
# Output:
#   Floating-point number in [0, 1).
#
# Note:
#   This uses 53 random bits, analogous to standard high-quality float output.
def RandomUnit():
    upperBits = ExtractUInt32() >> 5
    lowerBits = ExtractUInt32() >> 6
    return (upperBits * 67108864.0 + lowerBits) / 9007199254740992.0
# ###########################################################################


############################### Function 5 ####################################
# Name:
#   RandomUniform
#
# Purpose:
#   Return a pseudo-random number in [lowerBound, upperBound).
#
# Input:
#   lowerBound : Lower endpoint.
#   upperBound : Upper endpoint.
#
# Output:
#   Floating-point number in the requested interval.
def RandomUniform(lowerBound, upperBound):
    return lowerBound + (upperBound - lowerBound) * RandomUnit()
# ###########################################################################


############################### Function 6 ####################################
# Name:
#   WrapAngle
#
# Purpose:
#   Wrap an angle into (-pi, pi].
#
# Input:
#   angleValue : Any real angle.
#
# Output:
#   Wrapped angle.
def WrapAngle(angleValue):
    return ((angleValue + pi) % twoPi) - pi
# ###########################################################################


############################### Function 7 ####################################
# Name:
#   ApproxSin
#
# Purpose:
#   Approximate sin(x) with no math library.
#
# Method:
#   Uses a fast parabolic approximation with one correction step after reducing
#   the angle to (-pi, pi].
#
# Input:
#   angleValue : Real angle.
#
# Output:
#   Approximation of sin(angleValue).
def ApproxSin(angleValue):
    wrappedAngle = WrapAngle(angleValue)

    coefficientB = 4.0 / pi
    coefficientC = -4.0 / (pi * pi)

    sineValue = coefficientB * wrappedAngle + coefficientC * wrappedAngle * abs(wrappedAngle)

    correction = 0.225
    sineValue = correction * (sineValue * abs(sineValue) - sineValue) + sineValue

    return sineValue
# ###########################################################################


############################### Function 8 ####################################
# Name:
#   ApproxCos
#
# Purpose:
#   Approximate cos(x) with no math library.
#
# Input:
#   angleValue : Real angle.
#
# Output:
#   Approximation of cos(angleValue).
def ApproxCos(angleValue):
    return ApproxSin(angleValue + halfPi)
# ###########################################################################


############################### Function 9 ####################################
# Name:
#   ApproxSqrt
#
# Purpose:
#   Compute sqrt(x) with Newton iteration and no math library.
#
# Input:
#   value : Nonnegative real number.
#
# Output:
#   Approximation of sqrt(value).
def ApproxSqrt(value):
    if value <= 0.0:
        return 0.0

    if value < 1.0:
        estimate = 1.0
    else:
        estimate = value

    for _ in range(8):
        estimate = 0.5 * (estimate + value / estimate)

    return estimate
# ###########################################################################


############################### Function 10 ###################################
# Name:
#   GenerateNaturalFrequencies
#
# Purpose:
#   Generate omega_i uniformly on [pi/2, 3*pi].
#
# Input:
#   count : Number of oscillators.
#
# Output:
#   List of natural frequencies.
def GenerateNaturalFrequencies(count):
    naturalFrequencies = []

    for _ in range(count):
        naturalFrequency = RandomUniform(frequencyMin, frequencyMax)
        naturalFrequencies.append(naturalFrequency)

    return naturalFrequencies
# ###########################################################################


############################### Function 11 ###################################
# Name:
#   GenerateInitialPhases
#
# Purpose:
#   Generate a random initial phase vector in [0, 2*pi).
#
# Input:
#   count : Number of oscillators.
#
# Output:
#   List of initial phases.
def GenerateInitialPhases(count):
    phaseAngles = []

    for _ in range(count):
        phaseAngle = RandomUniform(0.0, twoPi)
        phaseAngles.append(phaseAngle)

    return phaseAngles
# ###########################################################################


############################### Function 12 ###################################
# Name:
#   ComputeOrderComponents
#
# Purpose:
#   Compute
#
#       C = (1 / N) * sum_j cos(theta_j),
#       S = (1 / N) * sum_j sin(theta_j),
#       r = sqrt(C^2 + S^2).
#
# Input:
#   phaseAngles : Current phase vector.
#
# Output:
#   cosineAverage       : C
#   sineAverage         : S
#   coherenceMagnitude  : r
def ComputeOrderComponents(phaseAngles):
    cosineSum = 0.0
    sineSum = 0.0
    oscillatorTotal = len(phaseAngles)

    for phaseAngle in phaseAngles:
        cosineSum += ApproxCos(phaseAngle)
        sineSum += ApproxSin(phaseAngle)

    cosineAverage = cosineSum / oscillatorTotal
    sineAverage = sineSum / oscillatorTotal
    coherenceMagnitude = ApproxSqrt(
        cosineAverage * cosineAverage + sineAverage * sineAverage
    )

    return cosineAverage, sineAverage, coherenceMagnitude
# ###########################################################################


############################### Function 13 ###################################
# Name:
#   KuramotoVelocity
#
# Purpose:
#   Evaluate equation (3.3) using the exact identity
#
#       r * sin(psi - theta_i) = S * cos(theta_i) - C * sin(theta_i).
#
# Input:
#   phaseAngle        : theta_i
#   naturalFrequency  : omega_i
#   couplingStrength  : K
#   cosineAverage     : C
#   sineAverage       : S
#
# Output:
#   Time derivative of theta_i.
def KuramotoVelocity(
    phaseAngle,
    naturalFrequency,
    couplingStrength,
    cosineAverage,
    sineAverage
):
    cosineTheta = ApproxCos(phaseAngle)
    sineTheta = ApproxSin(phaseAngle)

    meanFieldTerm = sineAverage * cosineTheta - cosineAverage * sineTheta

    return naturalFrequency + couplingStrength * meanFieldTerm
# ###########################################################################


############################### Function 14 ###################################
# Name:
#   RungeKuttaStep
#
# Purpose:
#   Advance one oscillator phase by one RK4 step using equation (3.3).
#
# Important detail:
#   The assignment says:
#     compute r e^(i psi),
#     update each theta_j by delta t via RK4 as per equation (3.3),
#     then calculate r e^(i psi) again and repeat.
#
#   Therefore C and S are treated as fixed during a single time step and are
#   recomputed after the full step is completed.
#
# Input:
#   phaseAngle       : Current theta_i
#   naturalFrequency : omega_i
#   couplingStrength : K
#   cosineAverage    : C
#   sineAverage      : S
#   stepSize         : delta t
#
# Output:
#   Updated phase in [0, 2*pi).
def RungeKuttaStep(
    phaseAngle,
    naturalFrequency,
    couplingStrength,
    cosineAverage,
    sineAverage,
    stepSize
):
    k1 = KuramotoVelocity(
        phaseAngle,
        naturalFrequency,
        couplingStrength,
        cosineAverage,
        sineAverage
    )

    k2 = KuramotoVelocity(
        phaseAngle + 0.5 * stepSize * k1,
        naturalFrequency,
        couplingStrength,
        cosineAverage,
        sineAverage
    )

    k3 = KuramotoVelocity(
        phaseAngle + 0.5 * stepSize * k2,
        naturalFrequency,
        couplingStrength,
        cosineAverage,
        sineAverage
    )

    k4 = KuramotoVelocity(
        phaseAngle + stepSize * k3,
        naturalFrequency,
        couplingStrength,
        cosineAverage,
        sineAverage
    )

    nextPhaseAngle = phaseAngle + (stepSize / 6.0) * (
        k1 + 2.0 * k2 + 2.0 * k3 + k4
    )

    return nextPhaseAngle % twoPi
# ###########################################################################


############################### Function 15 ###################################
# Name:
#   BuildCouplingValues
#
# Purpose:
#   Build the list
#   0.0, 0.1, 0.2, ..., 10.0
#
# Input:
#   None.
#
# Output:
#   List of K values.
#
# Note:
#   This is 100 values, because the interval is closed.
def BuildCouplingValues():
    couplingValues = []
    couplingCount = int((couplingEnd - couplingStart) / couplingStep + 0.5) + 1

    for index in range(couplingCount):
        couplingValue = couplingStart + index * couplingStep
        couplingValues.append(couplingValue)

    return couplingValues
# ###########################################################################


############################### Function 16 ###################################
# Name:
#   SimulateSingleCoupling
#
# Purpose:
#   Simulate the Kuramoto model for one fixed K from t = 0 to t = 20.
#
# Input:
#   couplingStrength   : Current K
#   naturalFrequencies : Fixed omega_i values
#
# Output:
#   Final coherence r(20), used as rInfinity.
def SimulateSingleCoupling(couplingStrength, naturalFrequencies):
    phaseAngles = GenerateInitialPhases(len(naturalFrequencies))

    for _ in range(stepCount):
        cosineAverage, sineAverage, _ = ComputeOrderComponents(phaseAngles)

        nextPhaseAngles = [0.0] * len(phaseAngles)

        for index in range(len(phaseAngles)):
            nextPhaseAngles[index] = RungeKuttaStep(
                phaseAngles[index],
                naturalFrequencies[index],
                couplingStrength,
                cosineAverage,
                sineAverage,
                timeStep
            )

        phaseAngles = nextPhaseAngles

    _, _, finalCoherenceMagnitude = ComputeOrderComponents(phaseAngles)
    return finalCoherenceMagnitude
# ###########################################################################


############################### Function 17 ###################################
# Name:
#   WriteDataFile
#
# Purpose:
#   Write the (K, rInfinity) data to a CSV-style text file.
#
# Input:
#   couplingValues  : List of K values
#   coherenceValues : Corresponding rInfinity values
#
# Output:
#   Data file written to disk.
def WriteDataFile(couplingValues, coherenceValues):
    with open(dataFilename, "w", encoding="utf-8") as dataFile:
        dataFile.write("K,rInfinity\n")

        for couplingValue, coherenceValue in zip(couplingValues, coherenceValues):
            dataFile.write(str(couplingValue) + "," + str(coherenceValue) + "\n")
# ###########################################################################


############################### Function 18 ###################################
# Name:
#   WriteSvgPlot
#
# Purpose:
#   Write an SVG graph of rInfinity versus K, styled to resemble Figure 3.
#
# Input:
#   couplingValues  : List of K values
#   coherenceValues : Corresponding rInfinity values
#
# Output:
#   SVG file written to disk.
def WriteSvgPlot(couplingValues, coherenceValues):
    width = 760
    height = 420
    leftMargin = 90
    rightMargin = 40
    topMargin = 30
    bottomMargin = 80

    xMin = couplingValues[0]
    xMax = couplingValues[-1]
    yMin = 0.0
    yMax = 1.05

    def MapX(xValue):
        usableWidth = width - leftMargin - rightMargin
        return leftMargin + usableWidth * (xValue - xMin) / (xMax - xMin)

    def MapY(yValue):
        usableHeight = height - topMargin - bottomMargin
        return height - bottomMargin - usableHeight * (yValue - yMin) / (yMax - yMin)

    pointStringPieces = []
    for xValue, yValue in zip(couplingValues, coherenceValues):
        pointStringPieces.append(str(MapX(xValue)) + "," + str(MapY(yValue)))
    pointString = " ".join(pointStringPieces)

    criticalCoupling = 5.0
    criticalX = MapX(criticalCoupling)
    unityY = MapY(1.0)
    axisY = MapY(0.0)

    svgLines = []
    svgLines.append('<svg xmlns="http://www.w3.org/2000/svg" width="' + str(width) +
                    '" height="' + str(height) + '">')
    svgLines.append('<rect x="0" y="0" width="100%" height="100%" fill="white"/>')

    svgLines.append('<line x1="' + str(leftMargin) + '" y1="' + str(axisY) +
                    '" x2="' + str(width - rightMargin) + '" y2="' + str(axisY) +
                    '" stroke="black" stroke-width="1.5"/>')

    svgLines.append('<line x1="' + str(leftMargin) + '" y1="' + str(topMargin) +
                    '" x2="' + str(leftMargin) + '" y2="' + str(axisY) +
                    '" stroke="black" stroke-width="1.5"/>')

    svgLines.append('<line x1="' + str(leftMargin) + '" y1="' + str(unityY) +
                    '" x2="' + str(width - rightMargin) + '" y2="' + str(unityY) +
                    '" stroke="#777777" stroke-width="1.2" stroke-dasharray="10,8"/>')

    svgLines.append('<line x1="' + str(criticalX) + '" y1="' + str(topMargin) +
                    '" x2="' + str(criticalX) + '" y2="' + str(axisY) +
                    '" stroke="#777777" stroke-width="1.2" stroke-dasharray="10,8"/>')

    svgLines.append('<polyline fill="none" stroke="black" stroke-width="2.2" points="' +
                    pointString + '"/>')

    svgLines.append('<text x="' + str(leftMargin - 24) + '" y="' + str(unityY + 5) +
                    '" font-size="20">1</text>')

    svgLines.append('<text x="' + str(leftMargin - 68) + '" y="' + str(MapY(0.5)) +
                    '" font-size="20">rInfinity</text>')

    svgLines.append('<text x="' + str(criticalX - 8) + '" y="' + str(axisY + 32) +
                    '" font-size="22">5</text>')

    svgLines.append('<text x="' + str(width - rightMargin - 10) + '" y="' +
                    str(axisY + 32) + '" font-size="22">K</text>')

    svgLines.append('</svg>')

    with open(figureFilename, "w", encoding="utf-8") as figureFile:
        figureFile.write("\n".join(svgLines))
# ###########################################################################


############################### Function 19 ###################################
# Name:
#   Main
#
# Purpose:
#   Run the full simulation and write the requested outputs.
#
# Input:
#   None.
#
# Output:
#   Printed progress, data file, and SVG figure.
def Main():
    SetSeed(randomSeed)

    couplingValues = BuildCouplingValues()

    # Generate omega_i once, then vary K and use a fresh random phase vector
    # for each K.
    naturalFrequencies = GenerateNaturalFrequencies(oscillatorCount)

    coherenceValues = []

    print("Starting Kuramoto Simulation:")
    print("oscillatorCount =", oscillatorCount)
    print("timeStep        =", timeStep)
    print("K values        =", len(couplingValues))
    print()

    for couplingValue in couplingValues:
        coherenceValue = SimulateSingleCoupling(couplingValue, naturalFrequencies)
        coherenceValues.append(coherenceValue)

        print(f"K = {couplingValue:.1f} | rInfinity ≈ {coherenceValue:.6f}")

    WriteDataFile(couplingValues, coherenceValues)
    WriteSvgPlot(couplingValues, coherenceValues)

    print()
    print("Simulation complete!")
    print("Data output :", dataFilename)
    print("Figure file :", figureFilename)
    print("Expected onset of synchronization is near K = 5.")
# ###########################################################################

################################### Execution #################################
if __name__ == "__main__":
    Main()
# ###########################################################################

Starting Kuramoto Simulation:
oscillatorCount = 10000
timeStep        = 0.1
K values        = 101

K = 0.0 | rInfinity ≈ 0.008481
K = 0.1 | rInfinity ≈ 0.004721
K = 0.2 | rInfinity ≈ 0.009209
K = 0.3 | rInfinity ≈ 0.015722
K = 0.4 | rInfinity ≈ 0.004436
K = 0.5 | rInfinity ≈ 0.007752
K = 0.6 | rInfinity ≈ 0.009498
K = 0.7 | rInfinity ≈ 0.008874
K = 0.8 | rInfinity ≈ 0.011146
K = 0.9 | rInfinity ≈ 0.009225
K = 1.0 | rInfinity ≈ 0.006759
K = 1.1 | rInfinity ≈ 0.028447
K = 1.2 | rInfinity ≈ 0.016073
K = 1.3 | rInfinity ≈ 0.009218
K = 1.4 | rInfinity ≈ 0.022607
K = 1.5 | rInfinity ≈ 0.004177
K = 1.6 | rInfinity ≈ 0.017613
K = 1.7 | rInfinity ≈ 0.008846
K = 1.8 | rInfinity ≈ 0.012577
K = 1.9 | rInfinity ≈ 0.006751
K = 2.0 | rInfinity ≈ 0.024621
K = 2.1 | rInfinity ≈ 0.007986
K = 2.2 | rInfinity ≈ 0.012048
K = 2.3 | rInfinity ≈ 0.023377
K = 2.4 | rInfinity ≈ 0.014062
K = 2.5 | rInfinity ≈ 0.025814
K = 2.6 | rInfinity ≈ 0.012271
K = 2.7 | rInfinity ≈ 0.013139
K = 2.8 | rInfinity ≈ 0.009727
K 